# Workshop 3: อ่านตัวอักษร (OCR)
ตัวอย่างงานจริง: อ่านวันหมดอายุ เลข LOT รหัสสินค้า
ขั้นตอน: **ภาพเทา → Threshold แบบ Otsu → Tesseract อ่านข้อความ**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))   # ให้หาไฟล์ vision_camera.py เจอ

import cv2
import numpy as np
from vision_camera import show, grab_one, img_path, put_label
print("OpenCV", cv2.__version__, "พร้อมใช้งาน")

In [ ]:
import pytesseract, shutil

# Windows: ถ้าหา tesseract ไม่เจอ ให้ชี้ไปที่ตำแหน่งติดตั้งมาตรฐาน
WIN_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
if shutil.which("tesseract") is None and os.path.exists(WIN_PATH):
    pytesseract.pytesseract.tesseract_cmd = WIN_PATH
print("Tesseract", pytesseract.get_tesseract_version())

## 1. เตรียมภาพ

In [ ]:
img = cv2.imread(img_path("ocr_sample.png"))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
T, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu เลือกค่า T ให้เอง =", T)
show(img, bw, titles=["ภาพต้นฉบับ", "หลัง Threshold (Otsu)"])

## 2. อ่านข้อความ

In [ ]:
import time

def read_text(img_bw, digits_only=False):
    config = "--psm 6"                       # มองเป็นบล็อกข้อความหลายบรรทัด
    min_conf = 40
    if digits_only:
        config += " -c tessedit_char_whitelist=0123456789-/."
        min_conf = 0                         # โหมดนี้ Tesseract มักรายงานความมั่นใจเป็น 0
    data = pytesseract.image_to_data(img_bw, config=config,
                                     output_type=pytesseract.Output.DICT)
    lines, boxes = {}, []
    for i, word in enumerate(data["text"]):
        if word.strip() and float(data["conf"][i]) >= min_conf:
            key = (data["block_num"][i], data["line_num"][i])
            lines.setdefault(key, []).append(word.strip())
            boxes.append((data["left"][i], data["top"][i], data["width"][i], data["height"][i]))
    return [" ".join(w) for _, w in sorted(lines.items())], boxes

t0 = time.perf_counter()
lines, boxes = read_text(bw)
ms = (time.perf_counter() - t0) * 1000

out = img.copy()
for x, y, w, h in boxes:
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 180, 0), 2)
for t in lines:
    print(t)
print(f"ใช้เวลา {ms:.0f} ms")
show(out, titles=["คำที่อ่านได้"])

## 3. อ่านเฉพาะตัวเลข

In [ ]:
lines, _ = read_text(bw, digits_only=True)
print(lines)

## ลองเอง (ระดับ 2): วันหมดอายุเลยหรือยัง
หาบรรทัดที่ขึ้นต้นด้วย `EXP` แล้วเทียบกับวันนี้ ถ้าเลยแล้ว = **NG**

In [ ]:
from datetime import date
import re

lines, _ = read_text(bw)
for t in lines:
    m = re.search(r"EXP\s*(\d{4})-(\d{2})-(\d{2})", t)
    if m:
        exp = date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        print("วันหมดอายุ:", exp, " ->", "OK" if exp >= date.today() else "NG (หมดอายุแล้ว)")

## ถ่ายจากกล้อง
เปลี่ยน `"file"` เป็น `"emu"` หรือ `"basler"` (Restart Kernel ก่อนสลับระหว่าง emu กับ basler)

In [ ]:
frame = grab_one("file", image="ocr_sample.png")
g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
_, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(read_text(b)[0])